<a href="https://colab.research.google.com/github/Tselovanska/machine-learning-course/blob/main/HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report


from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import TomekLinks



In [2]:
df = pd.read_csv('customer_segmentation_train.csv')
df

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
...,...,...,...,...,...,...,...,...,...,...,...
8063,464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D
8064,464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D
8065,465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D
8066,467299,Female,No,27,Yes,Healthcare,1.0,Low,4.0,Cat_6,B


In [3]:
#df.describe()
#df.info()
df.nunique()


,0
ID,8068
Gender,2
Ever_Married,2
Age,67
Graduated,2
Profession,9
Work_Experience,15
Spending_Score,3
Family_Size,9
Var_1,7


In [4]:
#пайплайн

#розбиття даних
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df['Segmentation']
)

# інпути і таргети
input_cols = list(train_df.columns.drop(['Segmentation', 'ID']))
target_col = 'Segmentation'

train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()

test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

# колонки
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()

ordinal_cols = ['Spending_Score']
onehot_cols = train_inputs.select_dtypes('object').columns.tolist()
onehot_cols.remove('Spending_Score')


#трансформери
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', MinMaxScaler())
])


ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', OrdinalEncoder(categories=[['Low', 'Average', 'High']]))
])

onehot_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', OneHotEncoder())
])

#препроцессор
preprocessor = ColumnTransformer(
    transformers=[
      ('num', numeric_transformer, numeric_cols),
      ('ordinal', ordinal_transformer, ordinal_cols),
      ('cat', onehot_transformer, onehot_cols)
    ]
)



**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [5]:
# SMOTENC
cat_cols = ordinal_cols + onehot_cols
cat_cols_indx = [train_inputs.columns.get_loc(col) for col in cat_cols]

imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')

train_inputs_imputed = train_inputs.copy()
train_inputs_imputed[numeric_cols] = imputer_num.fit_transform(train_inputs[numeric_cols])
train_inputs_imputed[cat_cols] = imputer_cat.fit_transform(train_inputs[cat_cols])

smote_nc = SMOTENC(categorical_features=cat_cols_indx,random_state=42)
X_smotenc, y_smotenc = smote_nc.fit_resample(train_inputs_imputed, train_targets)

#SMOTE_Tomek
X_smotenc_processed = preprocessor.fit_transform(X_smotenc, y_smotenc)

tomek = TomekLinks()
X_smotetomek, y_smotetomek = tomek.fit_resample(X_smotenc_processed, y_smotenc)

print("До SMOTENC:", train_targets.value_counts().to_dict())
print("Після SMOTENC:", y_smotenc.value_counts().to_dict())
print("Після SMOTENC + Tomek:", y_smotetomek.value_counts().to_dict())

До SMOTENC: {'D': 1814, 'A': 1578, 'C': 1576, 'B': 1486}
Після SMOTENC: {'A': 1814, 'B': 1814, 'C': 1814, 'D': 1814}
Після SMOTENC + Tomek: {'A': 1814, 'C': 1463, 'D': 1432, 'B': 1383}


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [8]:

X_train_orig = preprocessor.fit_transform(train_inputs, train_targets)
y_train_orig = train_targets

X_test_process = preprocessor.transform(test_inputs)
y_test = test_targets


X_train_smotenc = preprocessor.transform(X_smotenc)
y_train_smotenc = y_smotenc

X_train_smotetomek = X_smotetomek
y_train_smotetomek = y_smotetomek


datasets = {
    'Original': (X_train_orig, y_train_orig),
    'SMOTENC': (X_train_smotenc, y_train_smotenc),
    'SMOTE-Tomek': (X_train_smotetomek, y_train_smotetomek),
}

reports = {}

for name, (X_train, y_train) in datasets.items():
    model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42))
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test_process)

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred))
    print()

    reports[name] = classification_report(y_test, y_pred, output_dict=True)


=== Original ===
              precision    recall  f1-score   support

           A       0.41      0.45      0.43       394
           B       0.41      0.15      0.22       372
           C       0.49      0.65      0.56       394
           D       0.65      0.75      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.47      1614
weighted avg       0.50      0.51      0.49      1614


=== SMOTENC ===
              precision    recall  f1-score   support

           A       0.41      0.48      0.44       394
           B       0.42      0.22      0.29       372
           C       0.50      0.61      0.55       394
           D       0.67      0.71      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.51      0.52      0.50      1614


=== SMOTE-Tomek ===
              precision    recall  f1-score   support

           A       0

In [9]:
summary = pd.DataFrame({
    name: {
        'accuracy': r['accuracy'],
        'macro_f1': r['macro avg']['f1-score'],
        'weighted_f1': r['weighted avg']['f1-score'],
    }
    for name, r in reports.items()
}).T

summary

,accuracy,macro_f1,weighted_f1
Original,0.513011,0.474760,0.486434
SMOTENC,0.516109,0.492608,0.502656
SMOTE-Tomek,0.497522,0.459598,0.471452


1. Метрика: macro F1
Дисбаланс класів помірний, але щоб оцінити якість по всіх 4 сегментах рівноцінно, а не лише по найбільшому — обрала macro F1 замість accuracy.
2. Найкраща модель: SMOTENC
Macro F1: оригінал 0.47, SMOTENC 0.49, SMOTE-Tomek 0.46. SMOTENC показав найкращий результат за всіма трьома метриками. SMOTE-Tomek виявився гіршим навіть за незбалансовані дані.
3. Різниця між моделями несуттєва — гіпотеза
Дисбаланс тут слабкий (1814 проти 1486), тому навряд чи він головна причина. Реальна проблема — клас B: recall провальний у всіх трьох варіантах (0.09–0.22), модель систематично плутає його з сусідніми класами. Схоже, B погано відокремлюється лінійною межею від інших сегментів, а логістична регресія іншу межу побудувати не може. SMOTE-Tomek ситуацію лише погіршив — прибравши межові приклади B, він забрав саме ту інформацію, на якій модель могла б вчитися розрізняти цей клас.
